# SHIPP Stage 7A — ORS enrichment → Bronze raw responses

`silver_candidate_pairs` → ORS Matrix → `shipp_bronze.ors_route_responses` (append-only)

| Decision | Value (all in `config/settings.py`) |
|---|---|
| Endpoint | `POST {ORS_MATRIX_BASE}{ORS_PROFILE}`, metrics distance + duration, units km |
| Strategy | many-to-one: one call per request destination, ≤ `ORS_MAX_SOURCES_PER_CALL` listing origins |
| Cache key | `(origin_key, dest_key, profile)`, coordinates rounded to `ROUTE_KEY_DECIMALS` |
| Cache | skip keys already in `silver_routes` as `OK` or `NO_ROUTE` |
| Rate limit | `ORS_MIN_SECONDS_BETWEEN_CALLS` between calls, `ORS_MAX_CALLS_PER_RUN` cap |
| Retry | 429/5xx/network: `ORS_MAX_RETRIES` with exponential backoff, honours `Retry-After`; 4xx never retried |
| Raw preservation | every call's final raw body or error is appended to Bronze **before** parsing |

Client: `data_pipeline/ingestion/ors_client.py` (unit-tested, never raises, never logs the key).
**Clear state and outputs → Run all.** Every gate prints PASS or stops. Never loosen an assert. Only writer of `ors_route_responses`.

In [0]:
%run ../common/shipp_silver_lib

In [0]:
import uuid
from datetime import datetime, timezone

from data_pipeline.ingestion.ors_client import ORSClient
from data_pipeline.silver.routes import BRONZE_ROUTE_SCHEMA

BRONZE_ROUTE_RESPONSES = f"{CATALOG}.{BRONZE_SCHEMA}.ors_route_responses"

client = ORSClient(dbutils.secrets.get(scope=ORS_SECRET_SCOPE, key=ORS_SECRET_KEY))
spark.sql(f"CREATE TABLE IF NOT EXISTS {BRONZE_ROUTE_RESPONSES} ({BRONZE_ROUTE_SCHEMA}) USING DELTA")
print(client, "→", BRONZE_ROUTE_RESPONSES)

## Step 1 — Route keys not cached yet, grouped into many-to-one calls

In [0]:
CATALOG = "bootcamp_students"

BRONZE_SCHEMA = "shipp_bronze"
SILVER_SCHEMA = "shipp_silver"
GOLD_SCHEMA = "shipp_gold"

# Bronze history
BRONZE_LISTINGS_HISTORY = f"{CATALOG}.{BRONZE_SCHEMA}.lb_listings_history"
BRONZE_REQUESTS_HISTORY = f"{CATALOG}.{BRONZE_SCHEMA}.lb_requests_history"

# Bronze enrichment
BRONZE_ROUTE_RESPONSES = (
    f"{CATALOG}.{BRONZE_SCHEMA}.ors_route_responses"
)

# Silver
SILVER_LISTINGS = f"{CATALOG}.{SILVER_SCHEMA}.silver_listings"
SILVER_REQUESTS = f"{CATALOG}.{SILVER_SCHEMA}.silver_requests"

SILVER_CANDIDATE_PAIRS = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_candidate_pairs"
)

SILVER_ROUTES = f"{CATALOG}.{SILVER_SCHEMA}.silver_routes"

# Gold
GOLD_CANDIDATE_MATCHES = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_candidate_matches"
)

GOLD_LISTING_SEARCH_DOCS = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_listing_search_docs"
)

GOLD_MARKETPLACE_METRICS = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_marketplace_metrics"
)

# Pipeline observability
PIPELINE_RUN_LOG = (
    f"{CATALOG}.{GOLD_SCHEMA}.pipeline_run_log"
)

In [0]:
SILVER_CANDIDATE_PAIRS = f"{CATALOG}.{SILVER_SCHEMA}.silver_candidate_pairs"

keys = (spark.table(SILVER_CANDIDATE_PAIRS)
        .select("origin_key", "dest_key", "profile", "origin_lat", "origin_lon", "dest_lat", "dest_lon")
        .distinct())
total_keys = keys.count()
if table_exists(SILVER_ROUTES):
    cached = spark.table(SILVER_ROUTES).filter(F.col("route_status").isin("OK", "NO_ROUTE")) \
                  .select("origin_key", "dest_key", "profile")
    keys = keys.join(cached, ["origin_key", "dest_key", "profile"], "left_anti")
todo = keys.orderBy("dest_key", "origin_key").collect()
print(f"Route keys: {total_keys} | cached: {total_keys - len(todo)} | to fetch: {len(todo)}")

groups = {}
for r in todo:
    groups.setdefault((r["dest_key"], r["dest_lat"], r["dest_lon"]), []).append(r)
planned = [(dk, dlat, dlon, rows[i:i + ORS_MAX_SOURCES_PER_CALL])
           for (dk, dlat, dlon), rows in groups.items()
           for i in range(0, len(rows), ORS_MAX_SOURCES_PER_CALL)]
print("Planned calls:", len(planned), "| cap:", ORS_MAX_CALLS_PER_RUN)

## Step 2 — Call ORS and append every raw result to Bronze

In [0]:
bronze_rows, ok_calls, failed_calls = [], 0, 0
try:
    for n, (dest_key, dlat, dlon, origins) in enumerate(planned[:ORS_MAX_CALLS_PER_RUN], 1):
        result = client.matrix_many_to_one([(o["origin_lat"], o["origin_lon"]) for o in origins], (dlat, dlon))
        bronze_rows.append((
            str(uuid.uuid4()), datetime.now(timezone.utc).replace(tzinfo=None), ORS_PROFILE, dest_key,
            [o["origin_key"] for o in origins], result.request_json, result.http_status,
            result.response_body, result.error_message, result.attempts, result.latency_ms,
        ))
        if result.ok:
            ok_calls += 1
        else:
            failed_calls += 1
            print(f"  call {n} FAILED: {result.error_message}")
        time.sleep(ORS_MIN_SECONDS_BETWEEN_CALLS)
finally:
    if bronze_rows:  # persisted even if the loop is interrupted
        spark.createDataFrame(bronze_rows, BRONZE_ROUTE_SCHEMA).write.format("delta").mode("append") \
             .saveAsTable(BRONZE_ROUTE_RESPONSES)
    print(f"Appended {len(bronze_rows)} raw responses (ok={ok_calls}, failed={failed_calls})")

remaining = max(0, len(planned) - ORS_MAX_CALLS_PER_RUN)
if remaining:
    print(f"NOTE — {remaining} calls left for the next run.")

## Step 3 — Gate

In [0]:
assert not planned or ok_calls > 0, "FAILED: every ORS call failed — see error_message in Bronze (key, host, quota)."
if failed_calls:
    print(f"WARNING — {failed_calls} failed call(s); they are retried automatically next run.")
display(spark.table(BRONZE_ROUTE_RESPONSES)
        .select("called_at", "dest_key", F.size("origin_keys").alias("origins"), "http_status",
                "attempts", "latency_ms", "error_message")
        .orderBy(F.col("called_at").desc()).limit(20))

summary = {"table": BRONZE_ROUTE_RESPONSES, "calls_ok": ok_calls, "calls_failed": failed_calls,
           "calls_remaining": remaining, "status": "PASS"}
print(summary)
dbutils.notebook.exit(json.dumps(summary))